# HIGGS — Tahmin Notebook

`egitim.ipynb` tarafından kaydedilen `model.joblib` dosyasını yükler ve örnekler üzerinde tahmin yapar.

Model bir Pipeline'dır (ölçekleme + öznitelik seçimi + sınıflandırıcı), bu yüzden ham 28 özellik vermek yeterlidir.

**Veri kaynağı:** `data/higgs/HIGGS.csv.gz` (varsa), yoksa sentetik örnek

In [1]:
import os
import numpy as np
import pandas as pd
import joblib

## Ayarlar

In [2]:
HIGGS_PATH = os.environ.get("HIGGS_PATH", "data/higgs/HIGGS.csv.gz")
N_ORNEK    = 10000   # kaç örnek tahmin edilsin

FEATURES = [
    "lepton_pT", "lepton_eta", "lepton_phi", "missing_energy_magnitude",
    "missing_energy_phi", "jet1_pt", "jet1_eta", "jet1_phi", "jet1_b_tag",
    "jet2_pt", "jet2_eta", "jet2_phi", "jet2_b_tag", "jet3_pt", "jet3_eta",
    "jet3_phi", "jet3_b_tag", "jet4_pt", "jet4_eta", "jet4_phi", "jet4_b_tag",
    "m_jj", "m_jjj", "m_lv", "m_jlv", "m_bb", "m_wbb", "m_wwbb"
]

print(f"HIGGS_PATH : {HIGGS_PATH}")
print(f"Dosya var  : {os.path.exists(HIGGS_PATH)}")

HIGGS_PATH : data/higgs/HIGGS.csv.gz
Dosya var  : True


## Örnek Veri Hazırla

In [3]:
def ornek_veri(n=10, seed=42):
    """
    Tahmin için örnek getirir.
    Gerçek veri (HIGGS_PATH) varsa ondan rastgele n örnek alır.
    Yoksa eğitimdekiyle aynı dağılımdan sentetik örnek üretir.
    """
    if HIGGS_PATH and os.path.exists(HIGGS_PATH):
        print(f"[veri] Gerçek HIGGS'ten {n} örnek alınıyor...")
        df = pd.read_csv(HIGGS_PATH, header=None, names=["label"] + FEATURES)
        df = df.sample(n=n, random_state=7).reset_index(drop=True)
        return df[FEATURES].values, df["label"].values

    print(f"[veri] Gerçek veri yok -> sentetik {n} örnek üretiliyor...")
    from sklearn.datasets import make_classification
    X, y = make_classification(
        n_samples=4000, n_features=28, n_informative=14,
        n_redundant=6, weights=[0.47, 0.53], flip_y=0.08,
        class_sep=0.9, random_state=seed
    )
    for j in [0,3,5,9,13,17,21,22,23,24,25,26,27]:
        X[:, j] = np.expm1(np.abs(X[:, j]) / 2.0)
    return X[-n:], y[-n:]


X_ornek, y_gercek = ornek_veri(n=N_ORNEK)
print(f"Örnek şekli: {X_ornek.shape}")

[veri] Gerçek HIGGS'ten 10000 örnek alınıyor...


Örnek şekli: (10000, 28)


## Model Yükle ve Tahmin Yap

In [4]:
if not os.path.exists("model.joblib"):
    raise FileNotFoundError("model.joblib bulunamadı. Önce egitim.ipynb'i çalıştırın.")

paket = joblib.load("model.joblib")
model = paket["model"]
ad    = paket["model_adi"]
print(f"Yüklenen model: {ad}")
print(f"Pipeline adımları: {[s[0] for s in model.steps]}")

Yüklenen model: XGBoost
Pipeline adımları: ['scaler', 'select', 'clf']


In [5]:
tahmin = model.predict(X_ornek)

if hasattr(model, "predict_proba"):
    skor   = model.predict_proba(X_ornek)[:, 1]
    kolon  = "signal_olasılık"
else:
    skor   = model.decision_function(X_ornek)
    kolon  = "karar_skoru"

sonuc = pd.DataFrame({
    "tahmin": tahmin,
    kolon   : np.round(skor, 4),
    "gerçek": y_gercek,
    "doğru" : (tahmin == y_gercek).astype(int)
})

print("Tahmin sonuçları (0=background / 1=signal):")
sonuc

Tahmin sonuçları (0=background / 1=signal):


,tahmin,signal_olasılık,gerçek,doğru
0,0,0.2992,1.0,0
1,1,0.9827,0.0,0
2,0,0.1149,0.0,1
3,0,0.2491,1.0,0
4,1,0.8707,1.0,1
...,...,...,...,...
9995,0,0.2403,0.0,1
9996,0,0.3666,1.0,0
9997,0,0.3842,0.0,1
9998,0,0.0559,0.0,1


In [6]:
dogru = (tahmin == y_gercek).mean()
print(f"Örneklem doğruluğu: {dogru:.2%}  ({int(dogru*N_ORNEK)}/{N_ORNEK} doğru)")

Örneklem doğruluğu: 72.76%  (7276/10000 doğru)
